# 出行数据处理

**按照活动空间的划分方式，也将出行数据划分为前稳定期、过渡期、适应期、后稳定期四个时期（取平均值）**

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

导入数据

In [ ]:
move_data = pd.read_csv('../../autodl-fs/move_data.csv')
move_data.head()

,pid,year,week,avg_daily_dist,avg_daily_dur,avg_daily_slow_dist,avg_daily_slow_freq,avg_daily_tube_dist,avg_daily_tube_freq,home_year,home_week,work_year,work_week,time,home_time,work_time
0,263.0,2023,9,4546.945606,39.944853,688.33551,0.288571,3219.587778,0.232308,2024.0,26.0,-1.0,-1.0,0,69.0,-1.0
1,263.0,2023,10,5599.000000,37.450000,409.00000,0.140000,3219.587778,0.232308,2024.0,26.0,-1.0,-1.0,1,69.0,-1.0
2,263.0,2023,11,3608.430000,78.950000,818.00000,0.290000,3219.587778,0.232308,2024.0,26.0,-1.0,-1.0,2,69.0,-1.0
3,263.0,2023,12,5414.570000,38.550000,568.14000,0.290000,3219.587778,0.232308,2024.0,26.0,-1.0,-1.0,3,69.0,-1.0
4,263.0,2023,13,5688.710000,35.680000,512.57000,0.290000,3219.587778,0.232308,2024.0,26.0,-1.0,-1.0,4,69.0,-1.0


In [ ]:
# 导入已有的活动空间数据，用于筛样本
df = pd.read_csv('../data/act_space_features.csv')

# 计算df和variables的pid重合情况
df_pid_set = set(df['pid'])
move_pid_set = set(move_data['pid'])
df_pid_set = set(df['pid'])

print("df和move_data的pid重合情况:")
print(f"df中的pid数量: {len(df_pid_set)}")
print(f"move_data中的pid数量: {len(move_pid_set)}")
print(f"重合的pid数量: {len(df_pid_set.intersection(move_pid_set))}")

df和move_data的pid重合情况:
df中的pid数量: 84860
move_data中的pid数量: 179627
重合的pid数量: 84860


In [ ]:
# 筛选move_data中pid在df中的样本
move_data = move_data[move_data['pid'].isin(df_pid_set)].reset_index(drop=True)
move_data.head()

,pid,year,week,avg_daily_dist,avg_daily_dur,avg_daily_slow_dist,avg_daily_slow_freq,avg_daily_tube_dist,avg_daily_tube_freq,home_year,home_week,work_year,work_week,time,home_time,work_time
0,270.0,2023,9,4445.699459,27.52125,338.102857,0.161429,NaN,NaN,2023.0,51.0,-1.0,-1.0,0,42.0,-1.0
1,270.0,2023,10,4084.570000,25.25000,338.102857,0.161429,NaN,NaN,2023.0,51.0,-1.0,-1.0,1,42.0,-1.0
2,270.0,2023,11,534.290000,1.30000,338.102857,0.161429,NaN,NaN,2023.0,51.0,-1.0,-1.0,2,42.0,-1.0
3,270.0,2023,12,4445.699459,27.52125,338.102857,0.161429,NaN,NaN,2023.0,51.0,-1.0,-1.0,3,42.0,-1.0
4,270.0,2023,13,386.290000,17.70000,184.860000,0.140000,NaN,NaN,2023.0,51.0,-1.0,-1.0,4,42.0,-1.0


In [ ]:
# 以0补全缺失值
move_data = move_data.fillna(0)
move_data.head()

,pid,year,week,avg_daily_dist,avg_daily_dur,avg_daily_slow_dist,avg_daily_slow_freq,avg_daily_tube_dist,avg_daily_tube_freq,home_year,home_week,work_year,work_week,time,home_time,work_time
0,270.0,2023,9,4445.699459,27.52125,338.102857,0.161429,0.0,0.0,2023.0,51.0,-1.0,-1.0,0,42.0,-1.0
1,270.0,2023,10,4084.570000,25.25000,338.102857,0.161429,0.0,0.0,2023.0,51.0,-1.0,-1.0,1,42.0,-1.0
2,270.0,2023,11,534.290000,1.30000,338.102857,0.161429,0.0,0.0,2023.0,51.0,-1.0,-1.0,2,42.0,-1.0
3,270.0,2023,12,4445.699459,27.52125,338.102857,0.161429,0.0,0.0,2023.0,51.0,-1.0,-1.0,3,42.0,-1.0
4,270.0,2023,13,386.290000,17.70000,184.860000,0.140000,0.0,0.0,2023.0,51.0,-1.0,-1.0,4,42.0,-1.0


划分时期

In [ ]:
def vectorized_period_assignment(full_data):
    # 预计算每个pid的时间节点
    period_params = full_data[['pid', 'home_time', 'work_time']].drop_duplicates()
    
    # 计算三类情况的时间节点（向量化操作）
    mask_both = (period_params['home_time'] >= 0) & (period_params['work_time'] >= 0)
    mask_home = (period_params['home_time'] >= 0) & (period_params['work_time'] < 0)
    mask_work = (period_params['home_time'] < 0) & (period_params['work_time'] >= 0)

    period_params['period0_end'] = np.select(
        [mask_both, mask_home, mask_work],
        [
            np.minimum(period_params['home_time'], period_params['work_time']) - 4,
            period_params['home_time'] - 4,
            period_params['work_time'] - 4
        ],
        default=np.nan
    )
    
    period_params['period1_end'] = np.select(
        [mask_both, mask_home, mask_work],
        [
            np.maximum(period_params['home_time'], period_params['work_time']),
            period_params['home_time'],
            period_params['work_time']
        ],
        default=np.nan
    )
    
    period_params['period2_end'] = np.select(
        [mask_both, mask_home, mask_work],
        [
            np.maximum(period_params['home_time'], period_params['work_time']) + 8,
            period_params['home_time'] + 8,
            period_params['work_time'] + 8
        ],
        default=np.nan
    )

    # 合并到主数据
    full_data = full_data.merge(
        period_params[['pid', 'period0_end', 'period1_end', 'period2_end']], 
        on='pid', 
        how='left'
    )

    # 向量化时期分配
    conditions = [
        full_data['time'] < full_data['period0_end'],  # period 0
        (full_data['time'] >= full_data['period0_end']) & (full_data['time'] < full_data['period1_end']),  # period 1
        (full_data['time'] >= full_data['period1_end']) & (full_data['time'] < full_data['period2_end']),  # period 2
        full_data['time'] >= full_data['period2_end']  # period 3
    ]
    
    full_data['period'] = np.select(conditions, [0, 1, 2, 3], default=0)
    
    # 清理中间列
    full_data.drop(columns=['period0_end', 'period1_end', 'period2_end'], inplace=True)
    
    return full_data

In [ ]:
# 添加时期列
move_data_period = vectorized_period_assignment(move_data)
move_data_period.head()

,pid,year,week,avg_daily_dist,avg_daily_dur,avg_daily_slow_dist,avg_daily_slow_freq,avg_daily_tube_dist,avg_daily_tube_freq,home_year,home_week,work_year,work_week,time,home_time,work_time,period
0,270.0,2023,9,4445.699459,27.52125,338.102857,0.161429,0.0,0.0,2023.0,51.0,-1.0,-1.0,0,42.0,-1.0,0
1,270.0,2023,10,4084.570000,25.25000,338.102857,0.161429,0.0,0.0,2023.0,51.0,-1.0,-1.0,1,42.0,-1.0,0
2,270.0,2023,11,534.290000,1.30000,338.102857,0.161429,0.0,0.0,2023.0,51.0,-1.0,-1.0,2,42.0,-1.0,0
3,270.0,2023,12,4445.699459,27.52125,338.102857,0.161429,0.0,0.0,2023.0,51.0,-1.0,-1.0,3,42.0,-1.0,0
4,270.0,2023,13,386.290000,17.70000,184.860000,0.140000,0.0,0.0,2023.0,51.0,-1.0,-1.0,4,42.0,-1.0,0


In [ ]:
# 按pid和period分组计算平均值
period_means = move_data_period.groupby(['pid', 'period'])[
    ['avg_daily_dist', 'avg_daily_dur', 'avg_daily_slow_dist', 
     'avg_daily_slow_freq', 'avg_daily_tube_dist', 'avg_daily_tube_freq']
].mean().reset_index()

# 显示结果
period_means.head()


,pid,period,avg_daily_dist,avg_daily_dur,avg_daily_slow_dist,avg_daily_slow_freq,avg_daily_tube_dist,avg_daily_tube_freq
0,270.0,0,4332.332824,27.689803,337.480376,0.158609,0.000000,0.000000
1,270.0,1,5522.682500,25.920000,344.016429,0.188214,0.000000,0.000000
2,270.0,2,6664.765610,39.248571,322.172788,0.194327,0.000000,0.000000
3,270.0,3,8348.616071,23.522896,216.660769,0.197484,0.000000,0.000000
4,2753.0,0,4312.610137,41.559062,196.667176,0.198235,6516.841111,0.302222


In [ ]:
# 将数据从长格式转换为宽格式
period_means_wide = period_means.pivot(
    index='pid',
    columns='period'
)

# 重命名列名
new_columns = []
for col in period_means_wide.columns.levels[0]:
    if col != 'pid':
        for period in range(4):
            new_columns.append(f'{col}{period}')

period_means_wide.columns = new_columns
period_means_wide = period_means_wide.reset_index()

# 显示结果
period_means_wide.head()


,pid,avg_daily_dist0,avg_daily_dist1,avg_daily_dist2,avg_daily_dist3,avg_daily_dur0,avg_daily_dur1,avg_daily_dur2,avg_daily_dur3,avg_daily_slow_dist0,...,avg_daily_slow_freq2,avg_daily_slow_freq3,avg_daily_tube_dist0,avg_daily_tube_dist1,avg_daily_tube_dist2,avg_daily_tube_dist3,avg_daily_tube_freq0,avg_daily_tube_freq1,avg_daily_tube_freq2,avg_daily_tube_freq3
0,270.0,4332.332824,5522.682500,6664.76561,8348.616071,27.689803,25.920000,39.248571,23.522896,337.480376,...,0.194327,0.197484,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000
1,2753.0,4312.610137,1853.804477,4167.83080,8626.394000,41.559062,49.980380,30.191250,38.711964,196.667176,...,0.158750,0.162321,6516.841111,6516.841111,3817.016667,3817.016667,0.302222,0.302222,0.35000,0.350000
2,4100.0,18299.752429,16907.715000,15326.96500,14898.077758,73.297000,75.640000,95.368750,64.282571,200.009429,...,0.176699,0.215924,10402.511857,6796.715000,6265.231250,7152.244932,0.822429,0.642500,0.76750,1.058679
3,4658.0,6585.396508,19090.512120,6925.46500,9237.327542,28.466600,62.722926,23.745000,25.592222,259.790284,...,0.148523,0.155152,7890.081607,12649.038810,8455.105000,7646.896667,0.637765,0.950455,0.50825,0.443111
4,4942.0,12464.078636,15210.605000,6220.39250,5017.791695,54.830000,58.930000,33.686250,20.522239,273.642727,...,0.281620,0.225618,0.000000,0.000000,433.860000,433.860000,0.000000,0.000000,0.14000,0.140000


In [ ]:
# 根据pid关联两个数据框
df = df.merge(period_means_wide, on='pid', how='left')

print(df.isnull().sum())

pid                     0
area0                   0
area1                   0
area2                   0
area3                   0
                       ..
avg_daily_tube_dist3    0
avg_daily_tube_freq0    0
avg_daily_tube_freq1    0
avg_daily_tube_freq2    0
avg_daily_tube_freq3    0
Length: 85, dtype: int64


In [ ]:
# 保存数据
df.to_csv('../data/act_space_travel_features.csv', index=False)